# Phase 3 PPO Hyperparameter Tuning (Price-Only)

This notebook performs practical PPO tuning with time-aware validation.

Outputs:
- `results/phase3_tuning/price_only_tuning_trials.csv`
- `results/phase3_tuning/best_params_price_only.json`

Then `phase3_price_only_ppo.ipynb` can load the JSON automatically.


In [9]:
# Optional installs
%pip install gymnasium stable-baselines3 matplotlib



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from phase2_trading_env import make_env_from_processed, DEFAULT_TICKERS

try:
    from stable_baselines3 import PPO
    from stable_baselines3.common.vec_env import DummyVecEnv
    SB3_AVAILABLE = True
except ModuleNotFoundError:
    SB3_AVAILABLE = False
    print("stable-baselines3 missing. Install it first.")


In [3]:
# Config
TRAIN_PATH = "processed/train_dataset.csv"
FEATURE_CONFIG_PATH = "processed/feature_config.json"
OUT_DIR = Path("results/phase3_tuning")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRIALS_CSV = OUT_DIR / "price_only_tuning_trials.csv"
BEST_JSON = OUT_DIR / "best_params_price_only.json"

N_TRIALS = 12
SEEDS = [7, 42]
TIMESTEPS_PER_TRIAL = 20_000
VAL_SPLIT_DATE = "2021-01-01"  # train: < split, val: >= split

with open(FEATURE_CONFIG_PATH, "r") as f:
    cfg = json.load(f)
price_features = cfg["price_features"]


In [4]:
# Build time-aware train/validation datasets from train_dataset
full_train = pd.read_csv(TRAIN_PATH, parse_dates=["date"]).sort_values(["date", "ticker"])
val_split = pd.Timestamp(VAL_SPLIT_DATE)

sub_train = full_train[full_train["date"] < val_split].copy()
sub_val = full_train[full_train["date"] >= val_split].copy()

print("sub_train rows:", len(sub_train), "dates:", sub_train["date"].min(), "to", sub_train["date"].max())
print("sub_val rows:", len(sub_val), "dates:", sub_val["date"].min(), "to", sub_val["date"].max())

assert len(sub_train) > 0 and len(sub_val) > 0, "Train/val split produced empty set"


sub_train rows: 5784 dates: 2018-02-20 00:00:00 to 2020-12-31 00:00:00
sub_val rows: 2016 dates: 2021-01-04 00:00:00 to 2021-12-31 00:00:00


In [5]:
# Helpers

def build_env_from_df(df):
    # Temporary csv-based adapter via local temp file for compatibility with make_env_from_processed
    tmp_path = OUT_DIR / "_tmp_split.csv"
    df.to_csv(tmp_path, index=False)
    return make_env_from_processed(
        dataset_path=tmp_path,
        feature_config_path=FEATURE_CONFIG_PATH,
        tickers=DEFAULT_TICKERS,
        feature_columns_override=price_features,
        initial_cash=1_000_000.0,
        transaction_cost=0.001,
        trade_fraction=0.10,
        use_event_scaling=False,
    )


def run_backtest(model, env, seed=42):
    obs, info = env.reset(seed=seed)
    done = False
    values = [info["portfolio_value"]]
    daily = []

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, _, step_info = env.step(action)
        values.append(float(step_info["portfolio_value"]))
        daily.append(float(step_info["daily_return"]))

    ser = pd.Series(values)
    daily_ret = pd.Series(daily)
    return ser, daily_ret


def sharpe(daily_ret):
    s = daily_ret.std(ddof=1)
    if s <= 1e-12:
        return 0.0
    return float((daily_ret.mean() / s) * np.sqrt(252))


def max_drawdown(values):
    peak = values.cummax()
    dd = values / peak - 1.0
    return float(dd.min())


In [6]:
# Random search space
if not SB3_AVAILABLE:
    raise ModuleNotFoundError("stable-baselines3 not installed. Install and rerun.")

rng = np.random.default_rng(42)

def sample_params():
    return {
        "learning_rate": float(10 ** rng.uniform(-4.5, -3.0)),
        "n_steps": int(rng.choice([512, 1024, 2048, 4096])),
        "batch_size": int(rng.choice([64, 128, 256, 512])),
        "n_epochs": int(rng.choice([5, 10])),
        "gamma": float(rng.choice([0.97, 0.99, 0.995])),
        "gae_lambda": float(rng.choice([0.90, 0.95, 0.98])),
        "ent_coef": float(rng.choice([0.0, 0.001, 0.005, 0.01])),
        "clip_range": float(rng.choice([0.1, 0.2, 0.3])),
        "vf_coef": float(rng.choice([0.25, 0.5, 0.75])),
    }


In [7]:
# Run tuning
results = []

for trial in range(N_TRIALS):
    params = sample_params()
    seed_metrics = []

    for seed in SEEDS:
        env_train = DummyVecEnv([lambda: build_env_from_df(sub_train)])
        model = PPO(
            "MlpPolicy",
            env_train,
            verbose=0,
            seed=seed,
            **params,
        )
        model.learn(total_timesteps=TIMESTEPS_PER_TRIAL)

        env_val = build_env_from_df(sub_val)
        values, daily = run_backtest(model, env_val, seed=seed)
        creturn = float(values.iloc[-1] / values.iloc[0] - 1.0)
        s = sharpe(daily)
        mdd = max_drawdown(values)
        score = s + 2.0 * creturn - 0.5 * abs(mdd)
        seed_metrics.append((score, s, creturn, mdd))

    score_mean = float(np.mean([x[0] for x in seed_metrics]))
    sharpe_mean = float(np.mean([x[1] for x in seed_metrics]))
    ret_mean = float(np.mean([x[2] for x in seed_metrics]))
    mdd_mean = float(np.mean([x[3] for x in seed_metrics]))

    row = {"trial": trial, **params,
           "score_mean": score_mean,
           "sharpe_mean": sharpe_mean,
           "cumret_mean": ret_mean,
           "maxdd_mean": mdd_mean}
    results.append(row)
    print(f"trial={trial:02d} score={score_mean:.4f} sharpe={sharpe_mean:.4f} ret={ret_mean:.4f} mdd={mdd_mean:.4f}")

trials_df = pd.DataFrame(results).sort_values("score_mean", ascending=False).reset_index(drop=True)
trials_df.to_csv(TRIALS_CSV, index=False)
print("Saved trials:", TRIALS_CSV)
trials_df.head(10)


trial=00 score=-9.4120 sharpe=-9.6929 ret=0.1659 mdd=-0.1017
trial=01 score=-11.4462 sharpe=-11.5471 ret=0.0787 mdd=-0.1133
trial=02 score=-2.6314 sharpe=-2.6441 ret=0.0463 mdd=-0.1596
trial=03 score=-2.4412 sharpe=-2.5397 ret=0.0779 mdd=-0.1145
trial=04 score=-4.4785 sharpe=-4.7274 ret=0.1528 mdd=-0.1134
trial=05 score=-3.4212 sharpe=-3.3515 ret=0.0077 mdd=-0.1704
trial=06 score=-2.6109 sharpe=-2.8069 ret=0.1285 mdd=-0.1221
trial=07 score=-2.5886 sharpe=-2.7368 ret=0.1024 mdd=-0.1129
trial=08 score=-2.2431 sharpe=-2.4763 ret=0.1539 mdd=-0.1493
trial=09 score=-16.2709 sharpe=-16.6742 ret=0.2281 mdd=-0.1059
trial=10 score=-4.6708 sharpe=-4.8355 ret=0.1075 mdd=-0.1007
trial=11 score=-13.7853 sharpe=-13.9946 ret=0.1314 mdd=-0.1071
Saved trials: results/phase3_tuning/price_only_tuning_trials.csv


,trial,learning_rate,n_steps,batch_size,n_epochs,gamma,gae_lambda,ent_coef,clip_range,vf_coef,score_mean,sharpe_mean,cumret_mean,maxdd_mean
0,8,0.000143,512,512,10,0.995,0.90,0.001,0.3,0.75,-2.243120,-2.476315,0.153924,-0.149306
1,3,0.000069,512,256,10,0.970,0.98,0.010,0.1,0.50,-2.441167,-2.539725,0.077910,-0.114525
2,7,0.000061,1024,64,10,0.990,0.90,0.000,0.2,0.75,-2.588556,-2.736825,0.102361,-0.112908
3,6,0.000414,1024,512,5,0.970,0.98,0.001,0.1,0.50,-2.610850,-2.806862,0.128534,-0.122112
4,2,0.000114,512,512,10,0.990,0.95,0.010,0.2,0.50,-2.631437,-2.644142,0.046255,-0.159612
5,5,0.000062,1024,128,5,0.970,0.95,0.000,0.3,0.75,-3.421214,-3.351488,0.007726,-0.170355
6,4,0.000434,2048,128,5,0.995,0.95,0.010,0.3,0.75,-4.478477,-4.727415,0.152808,-0.113359
7,10,0.000063,4096,64,10,0.995,0.98,0.005,0.2,0.75,-4.670807,-4.835465,0.107493,-0.100657
8,0,0.000458,2048,128,5,0.995,0.90,0.005,0.1,0.25,-9.411977,-9.692904,0.165883,-0.101677
9,1,0.000919,2048,512,10,0.995,0.95,0.000,0.3,0.50,-11.446240,-11.547069,0.078731,-0.113267


In [8]:
# Save best params JSON for phase3_price_only_ppo.ipynb
best = trials_df.iloc[0].to_dict()
keys = [
    "learning_rate", "n_steps", "batch_size", "n_epochs",
    "gamma", "gae_lambda", "ent_coef", "clip_range", "vf_coef"
]
best_params = {k: best[k] for k in keys}

payload = {
    "source": "phase3_hyperparam_tuning.ipynb",
    "objective": "score_mean = sharpe + 2*cumret - 0.5*|max_drawdown|",
    "n_trials": int(N_TRIALS),
    "seeds": SEEDS,
    "timesteps_per_trial": int(TIMESTEPS_PER_TRIAL),
    "val_split_date": VAL_SPLIT_DATE,
    "best_params": best_params,
    "best_row": best,
}

with open(BEST_JSON, "w") as f:
    json.dump(payload, f, indent=2)

print("Saved best params:", BEST_JSON)
print(json.dumps(best_params, indent=2))


Saved best params: results/phase3_tuning/best_params_price_only.json
{
  "learning_rate": 0.00014312907952658717,
  "n_steps": 512.0,
  "batch_size": 512.0,
  "n_epochs": 10.0,
  "gamma": 0.995,
  "gae_lambda": 0.9,
  "ent_coef": 0.001,
  "clip_range": 0.3,
  "vf_coef": 0.75
}


## Next Step

Open `phase3_price_only_ppo.ipynb` and run it.
It will automatically read `results/phase3_tuning/best_params_price_only.json` (if present) and use tuned PPO hyperparameters.
